In [0]:
# # ============================================================
# # Customer Landing Validation
# # ============================================================

# import json
# from datetime import datetime, timezone



# # ------------------------------------------------------------
# # 1. Input parameter from ADF / Databricks Job
# # ------------------------------------------------------------

# dbutils.widgets.text(
#     "customer_path",
#     "abfss://landing@olistdev1.dfs.core.windows.net/SAP/Customer/olist_customers_dataset.csv"
# )
# #pipelinne run id param from adf
# dbutils.widgets.text(
#     "pipeline_run_id",
#     ""
# )


# customer_path = dbutils.widgets.get("customer_path")
# pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
# print(f"Processing file: {customer_path}")
# print(f"ADF Pipeline Run ID: {pipeline_run_id}")

# # ------------------------------------------------------------
# # 2. Read Customer Landing File
# # ------------------------------------------------------------

# df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(customer_path)
# )

# print("Customer file successfully read.")


# # ------------------------------------------------------------
# # 3. Validate Record Count
# # ------------------------------------------------------------

# customer_count = df.count()

# print(f"Customer record count: {customer_count}")


# # ------------------------------------------------------------
# # 4. Customer Data Validation
# # ------------------------------------------------------------

# from pyspark.sql.functions import col, trim, sum, when


# validation_status = "PASS"
# failed_validation = None
# validation_error = None


# try:

#     # --------------------------------------------------------
#     # 4.1 Validate Required Columns
#     # --------------------------------------------------------

#     required_columns = [
#         "customer_id",
#         "customer_unique_id",
#         "customer_zip_code_prefix",
#         "customer_city",
#         "customer_state"
#     ]

#     actual_columns = df.columns

#     missing_columns = [
#         column
#         for column in required_columns
#         if column not in actual_columns
#     ]
   
#     if missing_columns:

#         raise Exception(
#             f"CUSTOMER_SCHEMA_VALIDATION: FAIL - "
#             f"Missing columns: {missing_columns}"
#         )

#     print("CUSTOMER_SCHEMA_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.2 Validate Record Count
#     # --------------------------------------------------------

#     if customer_count == 0:

#         raise Exception(
#             "CUSTOMER_DATA_VALIDATION: FAIL - "
#             "Customer file contains zero records"
#         )

#     print("CUSTOMER_DATA_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.3 Validate Null / Blank Values
#     # --------------------------------------------------------

#     null_counts = df.select(
#         *[
#             sum(
#                 when(
#                     col(column).isNull() |
#                     (trim(col(column).cast("string")) == ""),
#                     1
#                 ).otherwise(0)
#             ).alias(column)
#             for column in required_columns
#         ]
#     ).collect()[0]

#     null_results = {
#         column: null_counts[column]
#         for column in required_columns
#     }

#     invalid_null_columns = {
#         column: count
#         for column, count in null_results.items()
#         if count > 0
#     }

#     if invalid_null_columns:

#         raise Exception(
#             f"CUSTOMER_NULL_VALIDATION: FAIL - "
#             f"Null or blank values found: "
#             f"{invalid_null_columns}"
#         )

#     print("CUSTOMER_NULL_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.4 Validate Duplicate Customer IDs
#     # --------------------------------------------------------

#     duplicate_customer_ids = (
#         df.groupBy("customer_id")
#           .count()
#           .filter(col("count") > 1)
#     )

#     duplicate_count = duplicate_customer_ids.count()

#     if duplicate_count > 0:

#         raise Exception(
#             f"CUSTOMER_DUPLICATE_VALIDATION: FAIL - "
#             f"{duplicate_count} duplicate customer_id values found"
#         )

#     print("CUSTOMER_DUPLICATE_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.5 Validate Customer State
#     # --------------------------------------------------------

#     valid_states = {
#         "AC", "AL", "AP", "AM", "BA", "CE", "DF",
#         "ES", "GO", "MA", "MT", "MS", "MG", "PA",
#         "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
#         "RO", "RR", "SC", "SP", "SE", "TO"
#     }

#     invalid_states = (
#         df.select("customer_state")
#           .withColumn(
#               "customer_state",
#               trim(col("customer_state"))
#           )
#           .filter(
#               ~col("customer_state").isin(valid_states)
#           )
#           .groupBy("customer_state")
#           .count()
#     )

#     invalid_state_count = invalid_states.count()

#     if invalid_state_count > 0:

#         invalid_state_values = [
#             row["customer_state"]
#             for row in (
#                 invalid_states
#                 .select("customer_state")
#                 .distinct()
#                 .collect()
#             )
#         ]

#         raise Exception(
#             f"CUSTOMER_STATE_VALIDATION: FAIL - "
#             f"Invalid customer_state values found: "
#             f"{invalid_state_values}"
#         )

#     print("CUSTOMER_STATE_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.6 Validate Customer ZIP Code Prefix
#     # --------------------------------------------------------

#     zip_values = (
#         df.select("customer_zip_code_prefix")
#           .withColumn(
#               "zip_code",
#               trim(col("customer_zip_code_prefix").cast("string"))
#           )
#     )

#     invalid_zip_codes = (
#         zip_values
#         .filter(
#             ~col("zip_code").rlike("^[0-9]{4,5}$")
#         )
#     )

#     invalid_zip_count = invalid_zip_codes.count()

#     if invalid_zip_count > 0:

#         invalid_zip_values = [
#             row["zip_code"]
#             for row in (
#                 invalid_zip_codes
#                 .select("zip_code")
#                 .distinct()
#                 .collect()
#             )
#         ]

#         raise Exception(
#             f"CUSTOMER_ZIP_VALIDATION: FAIL - "
#             f"Invalid customer_zip_code_prefix values found: "
#             f"{invalid_zip_values}"
#         )

#     print("CUSTOMER_ZIP_VALIDATION: PASS")


#     print("CUSTOMER_VALIDATION: ALL CHECKS PASSED")


# except Exception as e:

#     validation_status = "FAIL"
#     validation_error = str(e)

#     if "CUSTOMER_SCHEMA_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_SCHEMA_VALIDATION"

#     elif "CUSTOMER_DATA_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_DATA_VALIDATION"

#     elif "CUSTOMER_NULL_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_NULL_VALIDATION"

#     elif "CUSTOMER_DUPLICATE_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_DUPLICATE_VALIDATION"

#     elif "CUSTOMER_STATE_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_STATE_VALIDATION"

#     elif "CUSTOMER_ZIP_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_ZIP_VALIDATION"

#     else:
#         failed_validation = "UNKNOWN_VALIDATION_ERROR"

#     print(failed_validation + ": FAIL")
#     print(validation_error)

# audit_path = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/customer_validation_result.json"
# )

# # ------------------------------------------------------------
# # 5. Audit Result
# # ------------------------------------------------------------
# import json
# import uuid
# from datetime import datetime, timezone


# # ------------------------------------------------------------
# # 5. Audit Result
# # ------------------------------------------------------------

# # Generate unique execution ID
# run_id = pipeline_run_id

# # Execution timestamp
# validation_timestamp = datetime.now(timezone.utc).isoformat()

# # Audit metadata
# source_system = "SAP"
# entity = "Customer"

# # Build audit result
# result = {
#     "run_id": run_id,
#     "source_system": source_system,
#     "entity": entity,
#     "status": validation_status,
#     "failed_validation": failed_validation,
#     "error_message": validation_error,
#     "record_count": customer_count,
#     "source_file": customer_path,
#     "validation_timestamp": validation_timestamp
# }

# result_json = json.dumps(result)

# # ------------------------------------------------------------
# # 6. Write Audit Result to ADLS
# # ------------------------------------------------------------

# audit_directory = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/"
# )

# audit_path = (
#     audit_directory +
#     f"customer_validation_{run_id}.json"
# )

# dbutils.fs.put(
#     audit_path,
#     result_json,
#     overwrite=False
# )

# print(f"Validation result written to: {audit_path}")


# # ------------------------------------------------------------
# # 7. Display Sample Data
# # ------------------------------------------------------------

# display(df.limit(5))


# # ------------------------------------------------------------
# # 8. Return Result
# # ------------------------------------------------------------

# if validation_status == "FAIL":

#     raise Exception(
#         f"{failed_validation}: {validation_error}"
#     )

# dbutils.notebook.exit(result_json)

In [0]:
# import json

# result = {
#     "status": "PASS",
#     "record_count": customer_count,
#     "source_file": customer_path
# }

# dbutils.notebook.exit(json.dumps(result))

negative testing


we create a new file with bad data

In [0]:
# old_audit_path = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/customer_validation_result.json"
# )

# dbutils.fs.rm(old_audit_path, True)

# print("Old audit file removed.")

In [0]:
# %sql
# SELECT *
# FROM read_files(
#     'abfss://landing@olistdev1.dfs.core.windows.net/Audit/SAP/Customer/',
#     format => 'json'
# )
# ORDER BY validation_timestamp DESC;

In [0]:
# %sql
# SELECT *
# FROM read_files(
#   'abfss://landing@olistdev1.dfs.core.windows.net/Audit/SAP/Customer/',
#   format => 'json'
# )
# ORDER BY validation_timestamp DESC;

UPDATED CODE WHEN WE DEVELOPED LANDING-> BRONZE WHERE BRONZE IS DELTA WE CHANGED 8TH AND 9TH TOPICS

This means:

Validation FAIL
     ↓
Exception
     ↓
NO Bronze write

and:

Validation PASS
     ↓
Add ingestion_date
     ↓
Write Delta
> ### #      ↓

In [0]:
# # ============================================================
# # Customer Landing Validation
# # ============================================================

# import json
# from datetime import datetime, timezone



# # ------------------------------------------------------------
# # 1. Input parameter from ADF / Databricks Job
# # ------------------------------------------------------------

# dbutils.widgets.text(
#     "customer_path",
#     "abfss://landing@olistdev1.dfs.core.windows.net/SAP/Customer/olist_customers_dataset.csv"
# )
# #pipelinne run id param from adf
# dbutils.widgets.text(
#     "pipeline_run_id",
#     ""
# )


# customer_path = dbutils.widgets.get("customer_path")
# pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
# print(f"Processing file: {customer_path}")
# print(f"ADF Pipeline Run ID: {pipeline_run_id}")

# # ------------------------------------------------------------
# # 2. Read Customer Landing File
# # ------------------------------------------------------------

# df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(customer_path)
# )

# print("Customer file successfully read.")


# # ------------------------------------------------------------
# # 3. Validate Record Count
# # ------------------------------------------------------------

# customer_count = df.count()

# print(f"Customer record count: {customer_count}")


# # ------------------------------------------------------------
# # 4. Customer Data Validation
# # ------------------------------------------------------------

# from pyspark.sql.functions import col, trim, sum, when


# validation_status = "PASS"
# failed_validation = None
# validation_error = None


# try:

#     # --------------------------------------------------------
#     # 4.1 Validate Required Columns
#     # --------------------------------------------------------

#     required_columns = [
#         "customer_id",
#         "customer_unique_id",
#         "customer_zip_code_prefix",
#         "customer_city",
#         "customer_state"
#     ]

#     actual_columns = df.columns

#     missing_columns = [
#         column
#         for column in required_columns
#         if column not in actual_columns
#     ]
   
#     if missing_columns:

#         raise Exception(
#             f"CUSTOMER_SCHEMA_VALIDATION: FAIL - "
#             f"Missing columns: {missing_columns}"
#         )

#     print("CUSTOMER_SCHEMA_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.2 Validate Record Count
#     # --------------------------------------------------------

#     if customer_count == 0:

#         raise Exception(
#             "CUSTOMER_DATA_VALIDATION: FAIL - "
#             "Customer file contains zero records"
#         )

#     print("CUSTOMER_DATA_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.3 Validate Null / Blank Values
#     # --------------------------------------------------------

#     null_counts = df.select(
#         *[
#             sum(
#                 when(
#                     col(column).isNull() |
#                     (trim(col(column).cast("string")) == ""),
#                     1
#                 ).otherwise(0)
#             ).alias(column)
#             for column in required_columns
#         ]
#     ).collect()[0]

#     null_results = {
#         column: null_counts[column]
#         for column in required_columns
#     }

#     invalid_null_columns = {
#         column: count
#         for column, count in null_results.items()
#         if count > 0
#     }

#     if invalid_null_columns:

#         raise Exception(
#             f"CUSTOMER_NULL_VALIDATION: FAIL - "
#             f"Null or blank values found: "
#             f"{invalid_null_columns}"
#         )

#     print("CUSTOMER_NULL_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.4 Validate Duplicate Customer IDs
#     # --------------------------------------------------------

#     duplicate_customer_ids = (
#         df.groupBy("customer_id")
#           .count()
#           .filter(col("count") > 1)
#     )

#     duplicate_count = duplicate_customer_ids.count()

#     if duplicate_count > 0:

#         raise Exception(
#             f"CUSTOMER_DUPLICATE_VALIDATION: FAIL - "
#             f"{duplicate_count} duplicate customer_id values found"
#         )

#     print("CUSTOMER_DUPLICATE_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.5 Validate Customer State
#     # --------------------------------------------------------

#     valid_states = {
#         "AC", "AL", "AP", "AM", "BA", "CE", "DF",
#         "ES", "GO", "MA", "MT", "MS", "MG", "PA",
#         "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
#         "RO", "RR", "SC", "SP", "SE", "TO"
#     }

#     invalid_states = (
#         df.select("customer_state")
#           .withColumn(
#               "customer_state",
#               trim(col("customer_state"))
#           )
#           .filter(
#               ~col("customer_state").isin(valid_states)
#           )
#           .groupBy("customer_state")
#           .count()
#     )

#     invalid_state_count = invalid_states.count()

#     if invalid_state_count > 0:

#         invalid_state_values = [
#             row["customer_state"]
#             for row in (
#                 invalid_states
#                 .select("customer_state")
#                 .distinct()
#                 .collect()
#             )
#         ]

#         raise Exception(
#             f"CUSTOMER_STATE_VALIDATION: FAIL - "
#             f"Invalid customer_state values found: "
#             f"{invalid_state_values}"
#         )

#     print("CUSTOMER_STATE_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.6 Validate Customer ZIP Code Prefix
#     # --------------------------------------------------------

#     zip_values = (
#         df.select("customer_zip_code_prefix")
#           .withColumn(
#               "zip_code",
#               trim(col("customer_zip_code_prefix").cast("string"))
#           )
#     )

#     invalid_zip_codes = (
#         zip_values
#         .filter(
#             ~col("zip_code").rlike("^[0-9]{4,5}$")
#         )
#     )

#     invalid_zip_count = invalid_zip_codes.count()

#     if invalid_zip_count > 0:

#         invalid_zip_values = [
#             row["zip_code"]
#             for row in (
#                 invalid_zip_codes
#                 .select("zip_code")
#                 .distinct()
#                 .collect()
#             )
#         ]

#         raise Exception(
#             f"CUSTOMER_ZIP_VALIDATION: FAIL - "
#             f"Invalid customer_zip_code_prefix values found: "
#             f"{invalid_zip_values}"
#         )

#     print("CUSTOMER_ZIP_VALIDATION: PASS")


#     print("CUSTOMER_VALIDATION: ALL CHECKS PASSED")


# except Exception as e:

#     validation_status = "FAIL"
#     validation_error = str(e)

#     if "CUSTOMER_SCHEMA_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_SCHEMA_VALIDATION"

#     elif "CUSTOMER_DATA_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_DATA_VALIDATION"

#     elif "CUSTOMER_NULL_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_NULL_VALIDATION"

#     elif "CUSTOMER_DUPLICATE_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_DUPLICATE_VALIDATION"

#     elif "CUSTOMER_STATE_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_STATE_VALIDATION"

#     elif "CUSTOMER_ZIP_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_ZIP_VALIDATION"

#     else:
#         failed_validation = "UNKNOWN_VALIDATION_ERROR"

#     print(failed_validation + ": FAIL")
#     print(validation_error)

# audit_path = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/customer_validation_result.json"
# )

# # ------------------------------------------------------------
# # 5. Audit Result
# # ------------------------------------------------------------
# import json
# import uuid
# from datetime import datetime, timezone


# # ------------------------------------------------------------
# # 5. Audit Result
# # ------------------------------------------------------------

# # Generate unique execution ID
# run_id = pipeline_run_id

# # Execution timestamp
# validation_timestamp = datetime.now(timezone.utc).isoformat()

# # Audit metadata
# source_system = "SAP"
# entity = "Customer"

# # Build audit result
# result = {
#     "run_id": run_id,
#     "source_system": source_system,
#     "entity": entity,
#     "status": validation_status,
#     "failed_validation": failed_validation,
#     "error_message": validation_error,
#     "record_count": customer_count,
#     "source_file": customer_path,
#     "validation_timestamp": validation_timestamp
# }

# result_json = json.dumps(result)

# # ------------------------------------------------------------
# # 6. Write Audit Result to ADLS
# # ------------------------------------------------------------

# audit_directory = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/"
# )

# audit_path = (
#     audit_directory +
#     f"customer_validation_{run_id}.json"
# )

# dbutils.fs.put(
#     audit_path,
#     result_json,
#     overwrite=False
# )

# print(f"Validation result written to: {audit_path}")


# # ------------------------------------------------------------
# # 7. Display Sample Data
# # ------------------------------------------------------------

# display(df.limit(5))


# # 8. Write Validated Data to Bronze
# # ------------------------------------------------------------

# if validation_status == "FAIL":
#     raise Exception(
#         f"{failed_validation}: {validation_error}"
#     )

# from pyspark.sql.functions import lit, to_date
# from datetime import datetime, timezone

# bronze_path = (
#         "abfss://bronze@olistdev1.dfs.core.windows.net/"
#     "SAP/Customer"
# )

# ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# df_bronze = df.withColumn(
#     "ingestion_date",
#     to_date(lit(ingestion_date))
# )

# df_bronze.write \
#     .format("delta") \
#     .mode("append") \
#     .partitionBy("ingestion_date") \
#     .save(bronze_path)

# print(f"Bronze Delta write successful: {bronze_path}")
# print(f"Ingestion date: {ingestion_date}")
# print(f"Bronze record count: {df_bronze.count()}")

# # ------------------------------------------------------------
# # 9. Return Result
# # ------------------------------------------------------------

# dbutils.notebook.exit(result_json)

In [0]:
# display(
#     spark.read
#     .format("delta")
#     .load("abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer")
#     .limit(5)
# )

GIVING ACCESS FOR BRONZE TO UNITY -> THEN WE RUN ABOVE CODE

In [0]:
# dbutils.fs.rm(
#     "abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer/",
#     recurse=True
# )

# print("Empty Bronze Customer directory removed.")

We're reusing your existing:

cred_olist_adls

In [0]:
# %sql
# CREATE EXTERNAL LOCATION extloc_olist_bronze
# URL 'abfss://bronze@olistdev1.dfs.core.windows.net/'
# WITH (CREDENTIAL cred_olist_adls);

NOW WE DO PERMISSION GRANT - grant your Databricks user access to Bronze

In [0]:
%sql
-- GRANT READ FILES,
--       WRITE FILES
-- ON EXTERNAL LOCATION extloc_olist_bronze
-- TO `ayushmanpandita1999@gmail.com`;

In [0]:
#VERIFY DELTA TABLE EXISTS:

In [0]:
# display(
#     dbutils.fs.ls(
#         "abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer/"
#     )
# )

In [0]:
# test_delta_path = "abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer/_delta_test"

# test_df = spark.range(1)

# test_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save(test_delta_path)

# print("DELTA WRITE TEST SUCCESS")

In [0]:
# display(
#     dbutils.fs.ls(
#         "abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer/"
#     )
# )

In [0]:
# display(
#     dbutils.fs.ls(
#         "abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer/ingestion_date=2026-09-07/"
#     )
# )

In [0]:
# from pyspark.sql.functions import count, countDistinct

# bronze_df = (
#     spark.read
#     .format("delta")
#     .load("abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer")
# )

# today_df = bronze_df.filter("ingestion_date = '2026-09-07'")

# print("Total Bronze records today:", today_df.count())
# print("Distinct customer IDs today:", today_df.select("customer_id").distinct().count())

developing indempotency - creating control tables

testing hash

In [0]:
# from pyspark.sql.types import (
#     StructType,
#     StructField,
#     StringType,
#     TimestampType,
#     DateType
# )

# control_path = (
#     "abfss://bronze@olistdev1.dfs.core.windows.net/"
#     "Control/Ingestion_Control"
# )

# control_schema = StructType([
#     StructField("source_system", StringType(), True),
#     StructField("entity", StringType(), True),
#     StructField("file_hash", StringType(), True),
#     StructField("source_file", StringType(), True),
#     StructField("ingestion_date", DateType(), True),
#     StructField("pipeline_run_id", StringType(), True),
#     StructField("status", StringType(), True),
#     StructField("processed_timestamp", TimestampType(), True)
# ])

# empty_control_df = spark.createDataFrame([], control_schema)

# empty_control_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .save(control_path)

# print("INGESTION_CONTROL Delta table created successfully.")
# print(control_path)

In [0]:
# # ============================================================
# # Customer Landing Validation
# # ============================================================

# import json
# from datetime import datetime, timezone



# # ------------------------------------------------------------
# # 1. Input parameter from ADF / Databricks Job
# # ------------------------------------------------------------

# dbutils.widgets.text(
#     "customer_path",
#     "abfss://landing@olistdev1.dfs.core.windows.net/SAP/Customer/olist_customers_dataset.csv"
# )
# #pipelinne run id param from adf
# dbutils.widgets.text(
#     "pipeline_run_id",
#     ""
# )


# customer_path = dbutils.widgets.get("customer_path")
# pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
# print(f"Processing file: {customer_path}")
# print(f"ADF Pipeline Run ID: {pipeline_run_id}")

# # ------------------------------------------------------------
# # 2. Read Customer Landing File
# # ------------------------------------------------------------

# df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(customer_path)
# )

# print("Customer file successfully read.")

# # ------------------------------------------------------------
# # Calculate Source File Hash
# # ------------------------------------------------------------

# from pyspark.sql.functions import sha2, col

# file_hash = (
#     spark.read
#     .format("binaryFile")
#     .load(customer_path)
#     .select(
#         sha2(col("content"), 256).alias("file_hash")
#     )
#     .first()["file_hash"]
# )

# print(f"Source file SHA-256: {file_hash}")

# # ------------------------------------------------------------
# # 3. Validate Record Count
# # ------------------------------------------------------------

# customer_count = df.count()

# print(f"Customer record count: {customer_count}")


# # ------------------------------------------------------------
# # 4. Customer Data Validation
# # ------------------------------------------------------------

# from pyspark.sql.functions import col, trim, sum, when


# validation_status = "PASS"
# failed_validation = None
# validation_error = None


# try:

#     # --------------------------------------------------------
#     # 4.1 Validate Required Columns
#     # --------------------------------------------------------

#     required_columns = [
#         "customer_id",
#         "customer_unique_id",
#         "customer_zip_code_prefix",
#         "customer_city",
#         "customer_state"
#     ]

#     actual_columns = df.columns

#     missing_columns = [
#         column
#         for column in required_columns
#         if column not in actual_columns
#     ]
   
#     if missing_columns:

#         raise Exception(
#             f"CUSTOMER_SCHEMA_VALIDATION: FAIL - "
#             f"Missing columns: {missing_columns}"
#         )

#     print("CUSTOMER_SCHEMA_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.2 Validate Record Count
#     # --------------------------------------------------------

#     if customer_count == 0:

#         raise Exception(
#             "CUSTOMER_DATA_VALIDATION: FAIL - "
#             "Customer file contains zero records"
#         )

#     print("CUSTOMER_DATA_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.3 Validate Null / Blank Values
#     # --------------------------------------------------------

#     null_counts = df.select(
#         *[
#             sum(
#                 when(
#                     col(column).isNull() |
#                     (trim(col(column).cast("string")) == ""),
#                     1
#                 ).otherwise(0)
#             ).alias(column)
#             for column in required_columns
#         ]
#     ).collect()[0]

#     null_results = {
#         column: null_counts[column]
#         for column in required_columns
#     }

#     invalid_null_columns = {
#         column: count
#         for column, count in null_results.items()
#         if count > 0
#     }

#     if invalid_null_columns:

#         raise Exception(
#             f"CUSTOMER_NULL_VALIDATION: FAIL - "
#             f"Null or blank values found: "
#             f"{invalid_null_columns}"
#         )

#     print("CUSTOMER_NULL_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.4 Validate Duplicate Customer IDs
#     # --------------------------------------------------------

#     duplicate_customer_ids = (
#         df.groupBy("customer_id")
#           .count()
#           .filter(col("count") > 1)
#     )

#     duplicate_count = duplicate_customer_ids.count()

#     if duplicate_count > 0:

#         raise Exception(
#             f"CUSTOMER_DUPLICATE_VALIDATION: FAIL - "
#             f"{duplicate_count} duplicate customer_id values found"
#         )

#     print("CUSTOMER_DUPLICATE_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.5 Validate Customer State
#     # --------------------------------------------------------

#     valid_states = {
#         "AC", "AL", "AP", "AM", "BA", "CE", "DF",
#         "ES", "GO", "MA", "MT", "MS", "MG", "PA",
#         "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
#         "RO", "RR", "SC", "SP", "SE", "TO"
#     }

#     invalid_states = (
#         df.select("customer_state")
#           .withColumn(
#               "customer_state",
#               trim(col("customer_state"))
#           )
#           .filter(
#               ~col("customer_state").isin(valid_states)
#           )
#           .groupBy("customer_state")
#           .count()
#     )

#     invalid_state_count = invalid_states.count()

#     if invalid_state_count > 0:

#         invalid_state_values = [
#             row["customer_state"]
#             for row in (
#                 invalid_states
#                 .select("customer_state")
#                 .distinct()
#                 .collect()
#             )
#         ]

#         raise Exception(
#             f"CUSTOMER_STATE_VALIDATION: FAIL - "
#             f"Invalid customer_state values found: "
#             f"{invalid_state_values}"
#         )

#     print("CUSTOMER_STATE_VALIDATION: PASS")


#     # --------------------------------------------------------
#     # 4.6 Validate Customer ZIP Code Prefix
#     # --------------------------------------------------------

#     zip_values = (
#         df.select("customer_zip_code_prefix")
#           .withColumn(
#               "zip_code",
#               trim(col("customer_zip_code_prefix").cast("string"))
#           )
#     )

#     invalid_zip_codes = (
#         zip_values
#         .filter(
#             ~col("zip_code").rlike("^[0-9]{4,5}$")
#         )
#     )

#     invalid_zip_count = invalid_zip_codes.count()

#     if invalid_zip_count > 0:

#         invalid_zip_values = [
#             row["zip_code"]
#             for row in (
#                 invalid_zip_codes
#                 .select("zip_code")
#                 .distinct()
#                 .collect()
#             )
#         ]

#         raise Exception(
#             f"CUSTOMER_ZIP_VALIDATION: FAIL - "
#             f"Invalid customer_zip_code_prefix values found: "
#             f"{invalid_zip_values}"
#         )

#     print("CUSTOMER_ZIP_VALIDATION: PASS")


#     print("CUSTOMER_VALIDATION: ALL CHECKS PASSED")


# except Exception as e:

#     validation_status = "FAIL"
#     validation_error = str(e)

#     if "CUSTOMER_SCHEMA_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_SCHEMA_VALIDATION"

#     elif "CUSTOMER_DATA_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_DATA_VALIDATION"

#     elif "CUSTOMER_NULL_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_NULL_VALIDATION"

#     elif "CUSTOMER_DUPLICATE_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_DUPLICATE_VALIDATION"

#     elif "CUSTOMER_STATE_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_STATE_VALIDATION"

#     elif "CUSTOMER_ZIP_VALIDATION" in validation_error:
#         failed_validation = "CUSTOMER_ZIP_VALIDATION"

#     else:
#         failed_validation = "UNKNOWN_VALIDATION_ERROR"

#     print(failed_validation + ": FAIL")
#     print(validation_error)

# audit_path = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/customer_validation_result.json"
# )

# # ------------------------------------------------------------
# # 5. Audit Result
# # ------------------------------------------------------------
# import json
# import uuid
# from datetime import datetime, timezone


# # ------------------------------------------------------------
# # 5. Audit Result
# # ------------------------------------------------------------

# # Generate unique execution ID
# run_id = pipeline_run_id

# # Execution timestamp
# validation_timestamp = datetime.now(timezone.utc).isoformat()

# # Audit metadata
# source_system = "SAP"
# entity = "Customer"

# # Build audit result
# result = {
#     "run_id": run_id,
#     "source_system": source_system,
#     "entity": entity,
#     "status": validation_status,
#     "failed_validation": failed_validation,
#     "error_message": validation_error,
#     "record_count": customer_count,
#     "source_file": customer_path,
#     "validation_timestamp": validation_timestamp
# }

# result_json = json.dumps(result)

# # ------------------------------------------------------------
# # 6. Write Audit Result to ADLS
# # ------------------------------------------------------------

# audit_directory = (
#     "abfss://landing@olistdev1.dfs.core.windows.net/"
#     "Audit/SAP/Customer/"
# )

# audit_path = (
#     audit_directory +
#     f"customer_validation_{run_id}.json"
# )

# dbutils.fs.put(
#     audit_path,
#     result_json,
#     overwrite=False
# )

# print(f"Validation result written to: {audit_path}")


# # ------------------------------------------------------------
# # 7. Display Sample Data
# # ------------------------------------------------------------

# display(df.limit(5))


# # 8. Write Validated Data to Bronze
# # ------------------------------------------------------------

# if validation_status == "FAIL":
#     raise Exception(
#         f"{failed_validation}: {validation_error}"
#     )

# from pyspark.sql.functions import lit, to_date
# from datetime import datetime, timezone

# bronze_path = (
#         "abfss://bronze@olistdev1.dfs.core.windows.net/"
#     "SAP/Customer"
# )

# ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# df_bronze = df.withColumn(
#     "ingestion_date",
#     to_date(lit(ingestion_date))
# )

# df_bronze.write \
#     .format("delta") \
#     .mode("append") \
#     .partitionBy("ingestion_date") \
#     .save(bronze_path)

# print(f"Bronze Delta write successful: {bronze_path}")
# print(f"Ingestion date: {ingestion_date}")
# print(f"Bronze record count: {df_bronze.count()}")

# # ------------------------------------------------------------
# # 9. Return Result
# # ------------------------------------------------------------

# dbutils.notebook.exit(result_json)

code that fixes hash

In [0]:
# ============================================================
# Customer Landing Validation
# ============================================================

import json
from datetime import datetime, timezone



# ------------------------------------------------------------
# 1. Input parameter from ADF / Databricks Job
# ------------------------------------------------------------

dbutils.widgets.text(
    "customer_path",
    "abfss://landing@olistdev1.dfs.core.windows.net/SAP/Customer/olist_customers_dataset.csv"
)
#pipelinne run id param from adf
dbutils.widgets.text(
    "pipeline_run_id",
    ""
)


customer_path = dbutils.widgets.get("customer_path")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
print(f"Processing file: {customer_path}")
print(f"ADF Pipeline Run ID: {pipeline_run_id}")

# ------------------------------------------------------------
# 2. Read Customer Landing File
# ------------------------------------------------------------

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customer_path)
)

print("Customer file successfully read.")

# ------------------------------------------------------------
# Calculate Source File Hash
# ------------------------------------------------------------

from pyspark.sql.functions import sha2, col

file_hash = (
    spark.read
    .format("binaryFile")
    .load(customer_path)
    .select(
        sha2(col("content"), 256).alias("file_hash")
    )
    .first()["file_hash"]
)

print(f"Source file SHA-256: {file_hash}")

# ------------------------------------------------------------
# 3. Validate Record Count
# ------------------------------------------------------------

customer_count = df.count()

print(f"Customer record count: {customer_count}")


# ------------------------------------------------------------
# 4. Customer Data Validation
# ------------------------------------------------------------

from pyspark.sql.functions import col, trim, sum, when


validation_status = "PASS"
failed_validation = None
validation_error = None


try:

    # --------------------------------------------------------
    # 4.1 Validate Required Columns
    # --------------------------------------------------------

    required_columns = [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]

    actual_columns = df.columns

    missing_columns = [
        column
        for column in required_columns
        if column not in actual_columns
    ]
   
    if missing_columns:

        raise Exception(
            f"CUSTOMER_SCHEMA_VALIDATION: FAIL - "
            f"Missing columns: {missing_columns}"
        )

    print("CUSTOMER_SCHEMA_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.2 Validate Record Count
    # --------------------------------------------------------

    if customer_count == 0:

        raise Exception(
            "CUSTOMER_DATA_VALIDATION: FAIL - "
            "Customer file contains zero records"
        )

    print("CUSTOMER_DATA_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.3 Validate Null / Blank Values
    # --------------------------------------------------------

    null_counts = df.select(
        *[
            sum(
                when(
                    col(column).isNull() |
                    (trim(col(column).cast("string")) == ""),
                    1
                ).otherwise(0)
            ).alias(column)
            for column in required_columns
        ]
    ).collect()[0]

    null_results = {
        column: null_counts[column]
        for column in required_columns
    }

    invalid_null_columns = {
        column: count
        for column, count in null_results.items()
        if count > 0
    }

    if invalid_null_columns:

        raise Exception(
            f"CUSTOMER_NULL_VALIDATION: FAIL - "
            f"Null or blank values found: "
            f"{invalid_null_columns}"
        )

    print("CUSTOMER_NULL_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.4 Validate Duplicate Customer IDs
    # --------------------------------------------------------

    duplicate_customer_ids = (
        df.groupBy("customer_id")
          .count()
          .filter(col("count") > 1)
    )

    duplicate_count = duplicate_customer_ids.count()

    if duplicate_count > 0:

        raise Exception(
            f"CUSTOMER_DUPLICATE_VALIDATION: FAIL - "
            f"{duplicate_count} duplicate customer_id values found"
        )

    print("CUSTOMER_DUPLICATE_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.5 Validate Customer State
    # --------------------------------------------------------

    valid_states = {
        "AC", "AL", "AP", "AM", "BA", "CE", "DF",
        "ES", "GO", "MA", "MT", "MS", "MG", "PA",
        "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
        "RO", "RR", "SC", "SP", "SE", "TO"
    }

    invalid_states = (
        df.select("customer_state")
          .withColumn(
              "customer_state",
              trim(col("customer_state"))
          )
          .filter(
              ~col("customer_state").isin(valid_states)
          )
          .groupBy("customer_state")
          .count()
    )

    invalid_state_count = invalid_states.count()

    if invalid_state_count > 0:

        invalid_state_values = [
            row["customer_state"]
            for row in (
                invalid_states
                .select("customer_state")
                .distinct()
                .collect()
            )
        ]

        raise Exception(
            f"CUSTOMER_STATE_VALIDATION: FAIL - "
            f"Invalid customer_state values found: "
            f"{invalid_state_values}"
        )

    print("CUSTOMER_STATE_VALIDATION: PASS")


    # --------------------------------------------------------
    # 4.6 Validate Customer ZIP Code Prefix
    # --------------------------------------------------------

    zip_values = (
        df.select("customer_zip_code_prefix")
          .withColumn(
              "zip_code",
              trim(col("customer_zip_code_prefix").cast("string"))
          )
    )

    invalid_zip_codes = (
        zip_values
        .filter(
            ~col("zip_code").rlike("^[0-9]{4,5}$")
        )
    )

    invalid_zip_count = invalid_zip_codes.count()

    if invalid_zip_count > 0:

        invalid_zip_values = [
            row["zip_code"]
            for row in (
                invalid_zip_codes
                .select("zip_code")
                .distinct()
                .collect()
            )
        ]

        raise Exception(
            f"CUSTOMER_ZIP_VALIDATION: FAIL - "
            f"Invalid customer_zip_code_prefix values found: "
            f"{invalid_zip_values}"
        )

    print("CUSTOMER_ZIP_VALIDATION: PASS")


    print("CUSTOMER_VALIDATION: ALL CHECKS PASSED")


except Exception as e:

    validation_status = "FAIL"
    validation_error = str(e)

    if "CUSTOMER_SCHEMA_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_SCHEMA_VALIDATION"

    elif "CUSTOMER_DATA_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_DATA_VALIDATION"

    elif "CUSTOMER_NULL_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_NULL_VALIDATION"

    elif "CUSTOMER_DUPLICATE_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_DUPLICATE_VALIDATION"

    elif "CUSTOMER_STATE_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_STATE_VALIDATION"

    elif "CUSTOMER_ZIP_VALIDATION" in validation_error:
        failed_validation = "CUSTOMER_ZIP_VALIDATION"

    else:
        failed_validation = "UNKNOWN_VALIDATION_ERROR"

    print(failed_validation + ": FAIL")
    print(validation_error)

audit_path = (
    "abfss://landing@olistdev1.dfs.core.windows.net/"
    "Audit/SAP/Customer/customer_validation_result.json"
)

# ------------------------------------------------------------
# 5. Audit Result
# ------------------------------------------------------------
import json
import uuid
from datetime import datetime, timezone


# ------------------------------------------------------------
# 5. Audit Result
# ------------------------------------------------------------

# Generate unique execution ID
run_id = pipeline_run_id

# Execution timestamp
validation_timestamp = datetime.now(timezone.utc).isoformat()

# Audit metadata
source_system = "SAP"
entity = "Customer"

# Build audit result
result = {
    "run_id": run_id,
    "source_system": source_system,
    "entity": entity,
    "status": validation_status,
    "failed_validation": failed_validation,
    "error_message": validation_error,
    "record_count": customer_count,
    "source_file": customer_path,
    "validation_timestamp": validation_timestamp,
    "file_hash": file_hash
}

result_json = json.dumps(result)

# ------------------------------------------------------------
# 6. Write Audit Result to ADLS
# ------------------------------------------------------------
# ------------------------------------------------------------
# 6. Write Audit Result for Validation Failure
# ------------------------------------------------------------

audit_directory = (
    "abfss://landing@olistdev1.dfs.core.windows.net/"
    "Audit/SAP/Customer/"
)

audit_path = (
    audit_directory +
    f"customer_validation_{run_id}_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S%f')}.json"
)

# Validation failures must be audited before stopping the job.
if validation_status == "FAIL":

    result_json = json.dumps(result)

    dbutils.fs.put(
        audit_path,
        result_json,
        overwrite=False
    )

    print(
        f"Validation failure audit written to: {audit_path}"
    )

# ------------------------------------------------------------
# 7. Display Sample Data
# ------------------------------------------------------------

display(df.limit(5))


# 8. Write Validated Data to Bronze
# ------------------------------------------------------------
# ------------------------------------------------------------
# 8. Idempotency Check + Bronze Delta Write
# ------------------------------------------------------------

if validation_status == "FAIL":
    raise Exception(
        f"{failed_validation}: {validation_error}"
    )

from delta.tables import DeltaTable
from pyspark.sql.functions import lit, to_date
from datetime import datetime, timezone


# ------------------------------------------------------------
# 8.1 Paths and ingestion metadata
# ------------------------------------------------------------

bronze_path = (
    "abfss://bronze@olistdev1.dfs.core.windows.net/"
    "SAP/Customer"
)

control_path = (
    "abfss://bronze@olistdev1.dfs.core.windows.net/"
    "Control/Ingestion_Control"
)

ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")


# ------------------------------------------------------------
# 8.2 Check whether this exact file was already processed
# ------------------------------------------------------------

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)

already_processed = (
    control_df
    .filter(
        (col("source_system") == "SAP") &
        (col("entity") == "Customer") &
        (col("file_hash") == file_hash) &
        (col("status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)

print(f"File already processed: {already_processed}")


# ------------------------------------------------------------
# 8.3 Skip duplicate file
# ------------------------------------------------------------

if already_processed:

    print(
        "IDEMPOTENCY_CHECK: SKIP - "
        "This exact source file was already processed."
    )

    validation_status = "SKIPPED"

    result["status"] = "SKIPPED"
    result["error_message"] = (
        "Source file already processed. "
        "No Bronze write performed."
    )

    result_json = json.dumps(result)

else:

    # --------------------------------------------------------
    # 8.4 Prepare Bronze DataFrame
    # --------------------------------------------------------

    df_bronze = df.withColumn(
        "ingestion_date",
        to_date(lit(ingestion_date))
    )


    # --------------------------------------------------------
    # 8.5 Write to Bronze Delta
    # --------------------------------------------------------

    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .partitionBy("ingestion_date") \
        .save(bronze_path)

    print(
        f"Bronze Delta write successful: {bronze_path}"
    )

    print(
        f"Ingestion date: {ingestion_date}"
    )

    print(
        f"Bronze record count: {df_bronze.count()}"
    )


    # --------------------------------------------------------
    # 8.6 Record successful ingestion in control table
    # --------------------------------------------------------

    control_record = spark.createDataFrame(
        [
            (
                "SAP",
                "Customer",
                file_hash,
                customer_path,
                datetime.strptime(
                    ingestion_date,
                    "%Y-%m-%d"
                ).date(),
                pipeline_run_id,
                "SUCCESS",
                datetime.now(timezone.utc)
            )
        ],
        [
            "source_system",
            "entity",
            "file_hash",
            "source_file",
            "ingestion_date",
            "pipeline_run_id",
            "status",
            "processed_timestamp"
        ]
    )

    control_record.write \
        .format("delta") \
        .mode("append") \
        .save(control_path)

    print(
        "INGESTION_CONTROL: SUCCESS - "
        "File recorded as processed."
    )

    # ------------------------------------------------------------
# 8.7 Write Final Audit Result
# ------------------------------------------------------------

if not already_processed:
    result["status"] = "SUCCESS"

result_json = json.dumps(result)

dbutils.fs.put(
    audit_path,
    result_json,
    overwrite=False
)

print(
    f"Final audit result written to: {audit_path}"
)

# ------------------------------------------------------------
# 8.8 Return Final Result to ADF
# ------------------------------------------------------------

dbutils.notebook.exit(result_json)

In [0]:
# %sql
# SELECT *
# FROM delta.`abfss://bronze@olistdev1.dfs.core.windows.net/Control/Ingestion_Control`;

In [0]:
# bronze_df = (
#     spark.read
#     .format("delta")
#     .load("abfss://bronze@olistdev1.dfs.core.windows.net/SAP/Customer")
# )

# print("Active Bronze records:", bronze_df.count())

In [0]:
# display(
#     spark.read
#     .format("delta")
#     .load("abfss://bronze@olistdev1.dfs.core.windows.net/Control/Ingestion_Control")
# )